# 02 · Answer: search, ground, cite

**You'll learn:** exactly what happens between typing a question and getting a cited answer, one step at a time, with the real data.

**Time:** about 10 minutes · **Cost:** about 2¢ · **Needs:** an API key in `.env`, and Docker Qdrant running or `USE_EMBEDDED_QDRANT = True`. **Runs on its own:** the second cell builds any missing data.

```mermaid
flowchart LR
  Q["1 · Question"] --> D["2a · Dense search<br/>meaning"]
  Q --> S["2b · Sparse search<br/>exact words"]
  D --> F["3 · RRF fusion"]
  S --> F
  F --> K["4 · Top-k = 8 chunks"]
  K --> P["5 · Grounded prompt"]
  P --> L["6 · LLM"]
  L --> C["7 · Parse CITED"]
  C -->|"cited ≥ 1"| A["8 · Answer + citations"]
  C -->|"cited none"| N["8 · 'Not found'"]
```

| Part | Steps | Code |
|---|---|---|
| 1 | Setup | |
| 2 | Retrieval: question, dense search, sparse search, fusion, top-k (steps 1–4) | `core/retrieval.py` |
| 3 | Generation: prompt, LLM, citations (steps 5–7) | `core/rag.py`, `core/llm.py` |
| 4 | The "not found" guardrail (step 8) | `core/rag.py` |
| 5 | Recap and your turn | |

---
## Part 1 · Setup

### Setup
The next cell connects the notebook to the project. Read its first lines: `USE_EMBEDDED_QDRANT` picks where the database runs (Docker server, or embedded with no Docker). It prints which LLM, embedding model and database this run uses, all taken from your `.env`.

In [1]:
# ── Settings ────────────────────────────────────────────────────────────
# False: use the Qdrant server from `docker compose up -d`.
# True:  no Docker. Qdrant runs inside this notebook and stores data in data/qdrant_local/.
#        Restart the kernel after changing this, and run one notebook at a time
#        (only one process can open the embedded folder).
USE_EMBEDDED_QDRANT = False

# Setup: run from the repo root so `core`/`ingestion` imports and data/ paths work.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)
if USE_EMBEDDED_QDRANT:
    os.environ["QDRANT_PATH"] = "data/qdrant_local"
warnings.filterwarnings("ignore", message="Qdrant client version")  # harmless client/server version notice

import pandas as pd

import core  # noqa: F401  (loads .env)
import core.embeddings as emb
import core.llm as llm
from core.clients import COLLECTION_NAME, QDRANT_PATH, QDRANT_URL, get_qdrant_client

pd.set_option("display.max_colwidth", 120)
print(f"LLM:        {llm.PROVIDER} / {llm.MODEL}")
print(f"Embeddings: {emb.PROVIDER} / {emb.MODEL}")
print(f"Qdrant:     {'embedded @ ' + QDRANT_PATH if QDRANT_PATH else 'server @ ' + QDRANT_URL}")

LLM:        gemini / gemini-3.5-flash
Embeddings: gemini / gemini-embedding-001
Qdrant:     server @ http://localhost:6333


In [2]:
# Is the vector database running?
try:
    names = [c.name for c in get_qdrant_client().get_collections().collections]
    print("Qdrant is up. Collections:", names or "(none yet)")
except Exception as exc:
    raise SystemExit(
        "Qdrant is not reachable. Either open Docker Desktop and run `docker compose up -d`, "
        "or set USE_EMBEDDED_QDRANT = True in the first cell and restart the kernel."
    ) from exc

# Make sure the data is ready, so this notebook runs on its own. prepare_data() fetches,
# chunks and indexes only what's missing: a few minutes the very first time, then seconds.
from ingestion.prepare import prepare_data

for name, count in prepare_data().items():
    print(f"{name}: {count} points")

Qdrant is up. Collections: ['sec_filings_advanced', 'sec_filings_advanced_gemini', 'sec_filings_basic', 'sec_filings_basic_gemini']
sec_filings_basic_gemini: 168 points
sec_filings_advanced_gemini: 310 points


---
## Part 2 · Retrieval (steps 1–4)

### Step 1 · The question

This question needs one specific paragraph from Item 7. Change it later and re-run the notebook.

In [3]:
QUESTION = "Why did Greater China net sales decrease in fiscal 2024?"
TOP_K = 8

### Step 2a · Dense search (meaning)

The question is embedded with the same model as the chunks, and Qdrant returns the chunks whose vectors point in the most similar direction. **Score = cosine similarity**: higher is closer, and values usually fall between 0.2 and 0.8.

`hybrid_search()` fetches **2 × k = 16** candidates from each search before fusing, so we fetch 16 here too and show the first 8.

In [4]:
from core.retrieval import embed_query_dense, embed_query_sparse

client = get_qdrant_client()
CANDIDATES = TOP_K * 2


def as_table(points):
    return pd.DataFrame([
        {
            "rank": i + 1,
            "id": p.id,
            "score": round(p.score, 4),
            "section": p.payload["section"][:60],
            "text": p.payload["text"][:110].replace("\n", " "),
        }
        for i, p in enumerate(points)
    ])


dense_hits = client.query_points(
    COLLECTION_NAME, query=embed_query_dense(QUESTION), using="dense", limit=CANDIDATES
).points
as_table(dense_hits).head(TOP_K)

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,rank,id,score,section,text
0,1,84,0.8345,Item 7. Management's Discussion and Analysis of Financial,Greater China Greater China net sales decreased during 2024 compared to 2023 due primarily to lower net sales
1,2,81,0.7565,Item 7. Management's Discussion and Analysis of Financial,"Segment Operating Performance The following table shows net sales by reportable segment for 2024, 2023 and 202"
2,3,86,0.7529,Item 7. Management's Discussion and Analysis of Financial,Rest of Asia Pacific Rest of Asia Pacific net sales increased during 2024 compared to 2023 due primarily to hi
3,4,167,0.7415,Item 8. Financial Statements and Supplementary Data > Not,"Note 13 - Segment Information and Geographic Data The following tables show net sales for 2024, 2023 and"
4,5,91,0.7354,Item 7. Management's Discussion and Analysis of Financial,"Wearables, Home and Accessories Wearables, Home and Accessories net sales decreased during 2024 compared to 20"
5,6,82,0.7275,Item 7. Management's Discussion and Analysis of Financial,Americas Americas net sales increased during 2024 compared to 2023 due primarily to higher net sales of Servic
6,7,85,0.7186,Item 7. Management's Discussion and Analysis of Financial,Japan Japan net sales increased during 2024 compared to 2023 due primarily to higher net sales of iPhone. The
7,8,166,0.7062,Item 8. Financial Statements and Supplementary Data > Not,Note 13 - Segment Information and Geographic Data The Company manages its business primarily on a geographic b


### Step 2b · Sparse search (exact words)

The question becomes a BM25 sparse vector and is matched against each chunk's sparse vector. **Score = BM25**: it has no fixed upper limit, so don't compare these numbers with the dense scores. Chunks that literally contain "Greater China" and "net sales" do well here.

In [7]:
sparse_hits = client.query_points(
    COLLECTION_NAME, query=embed_query_sparse(QUESTION), using="sparse", limit=CANDIDATES
).points
as_table(sparse_hits).head(TOP_K)

,rank,id,score,section,text
0,1,84,18.4890,Item 7. Management's Discussion and Analysis of Financial,Greater China Greater China net sales decreased during 2024 compared to 2023 due primarily to lower net sales
1,2,166,14.3317,Item 8. Financial Statements and Supplementary Data > Not,Note 13 - Segment Information and Geographic Data The Company manages its business primarily on a geographic b
2,3,134,13.3674,Item 8. Financial Statements and Supplementary Data > Not,Note 2 - Revenue For the sale of third-party products where the Company obtains control of the product before
3,4,81,13.1368,Item 7. Management's Discussion and Analysis of Financial,"Segment Operating Performance The following table shows net sales by reportable segment for 2024, 2023 and 202"
4,5,167,12.0820,Item 8. Financial Statements and Supplementary Data > Not,"Note 13 - Segment Information and Geographic Data The following tables show net sales for 2024, 2023 and"
5,6,90,11.7165,Item 7. Management's Discussion and Analysis of Financial,iPad iPad net sales decreased during 2024 compared to 2023 due primarily to lower net sales of iPad Pro and th
6,7,91,11.5220,Item 7. Management's Discussion and Analysis of Financial,"Wearables, Home and Accessories Wearables, Home and Accessories net sales decreased during 2024 compared to 20"
7,8,86,9.5227,Item 7. Management's Discussion and Analysis of Financial,Rest of Asia Pacific Rest of Asia Pacific net sales increased during 2024 compared to 2023 due primarily to hi


How much do the two lists agree? Chunks found by both are strong candidates; chunks found by only one are what hybrid search adds.

In [8]:
dense_ids = [p.id for p in dense_hits]
sparse_ids = [p.id for p in sparse_hits]
both = set(dense_ids) & set(sparse_ids)
print(f"found by both: {len(both)} · dense only: {len(set(dense_ids) - both)} · sparse only: {len(set(sparse_ids) - both)}  (of {CANDIDATES} each)")

found by both: 13 · dense only: 3 · sparse only: 3  (of 16 each)


### Step 3 · RRF fusion, by hand and by Qdrant

**Reciprocal Rank Fusion** scores each chunk by its *rank* in each list, `Σ 1 / (k + rank)`, so the two kinds of scores never need to be compared. First by hand with the classic `k = 60`:

In [10]:
K = 60
rrf = {}
for ranking in (dense_ids, sparse_ids):
    for rank, chunk_id in enumerate(ranking, start=1):
        rrf[chunk_id] = rrf.get(chunk_id, 0) + 1 / (K + rank)
manual_top = sorted(rrf, key=lambda i: -rrf[i])[:TOP_K]

pd.DataFrame([
    {
        "rank": n + 1,
        "id": i,
        "dense rank": dense_ids.index(i) + 1 if i in dense_ids else "-",
        "sparse rank": sparse_ids.index(i) + 1 if i in sparse_ids else "-",
        "RRF score": round(rrf[i], 4),
    }
    for n, i in enumerate(manual_top)
])

,rank,id,dense rank,sparse rank,RRF score
0,1,84,1,1,0.0328
1,2,81,2,4,0.0318
2,3,167,4,5,0.0310
3,4,166,8,2,0.0308
4,5,86,3,8,0.0306
5,6,91,5,7,0.0303
6,7,85,7,9,0.0294
7,8,90,12,6,0.0290


### Step 4 · Top-k: what the pipeline actually keeps

`hybrid_search()` does steps 2a, 2b and 3 in **one** Qdrant call (two `Prefetch` searches plus `FusionQuery(fusion=Fusion.RRF)`) and keeps the best `TOP_K = 8`. Qdrant uses its own constant inside RRF, so its scores differ from ours, but the idea is the same; the last line shows how many chunks both versions agree on.

In [11]:
from core.retrieval import hybrid_search

chunks = hybrid_search(QUESTION, top_k=TOP_K)


def found_by(chunk_id):
    sources = [name for name, ids in (("dense", dense_ids), ("sparse", sparse_ids)) if chunk_id in ids]
    return " + ".join(sources)


display(pd.DataFrame([
    {
        "rank": i + 1,
        "qdrant RRF score": round(c.score, 4),
        "found_by": found_by(c.id),
        "section": c.section[:60],
        "text": c.text[:100].replace("\n", " "),
    }
    for i, c in enumerate(chunks)
]))
print(f"Same chunks as the hand-made top {TOP_K}: {len(set(manual_top) & {c.id for c in chunks})} of {TOP_K}")

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,rank,qdrant RRF score,found_by,section,text
0,1,1.0000,dense + sparse,Item 7. Management's Discussion and Analysis of Financial,Greater China Greater China net sales decreased during 2024 compared to 2023 due primarily to lower
1,2,0.5333,dense + sparse,Item 7. Management's Discussion and Analysis of Financial,"Segment Operating Performance The following table shows net sales by reportable segment for 2024, 20"
2,3,0.4444,dense + sparse,Item 8. Financial Statements and Supplementary Data > Not,Note 13 - Segment Information and Geographic Data The Company manages its business primarily on a ge
3,4,0.3667,dense + sparse,Item 8. Financial Statements and Supplementary Data > Not,"Note 13 - Segment Information and Geographic Data The following tables show net sales for 2024, 2"
4,5,0.3611,dense + sparse,Item 7. Management's Discussion and Analysis of Financial,Rest of Asia Pacific Rest of Asia Pacific net sales increased during 2024 compared to 2023 due prima
5,6,0.3088,dense + sparse,Item 8. Financial Statements and Supplementary Data > Not,Note 2 - Revenue For the sale of third-party products where the Company obtains control of the produ
6,7,0.2917,dense + sparse,Item 7. Management's Discussion and Analysis of Financial,"Wearables, Home and Accessories Wearables, Home and Accessories net sales decreased during 2024 comp"
7,8,0.2250,dense + sparse,Item 7. Management's Discussion and Analysis of Financial,Japan Japan net sales increased during 2024 compared to 2023 due primarily to higher net sales of iP


Same chunks as the hand-made top 8: 7 of 8


**Choosing k.** More chunks give the LLM more context but cost more tokens and can add noise. This cell builds the prompt for a few values of k (no LLM call) to show the token cost:

In [12]:
import tiktoken

from core.rag import _build_prompt

enc = tiktoken.get_encoding("cl100k_base")
pd.DataFrame([
    {"top_k": k, "prompt tokens": len(enc.encode(_build_prompt(QUESTION, hybrid_search(QUESTION, top_k=k))))}
    for k in (3, 8, 16)
])

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,top_k,prompt tokens
0,3,808
1,8,2568
2,16,3476


---
## Part 3 · Generation (steps 5–7)

### Step 5 · The grounded prompt

`_build_prompt()` numbers the chunks `[0]…[7]`, labels each with company, section and fiscal year, and gives three instructions: answer **only** from the excerpts; end with `CITED: <numbers>`; if the excerpts aren't enough, say so and leave `CITED:` empty. Here is the beginning and the end of the real prompt:

In [13]:
prompt = _build_prompt(QUESTION, chunks)
print(len(enc.encode(prompt)), "tokens\n")
print(prompt[:700], "\n\n[... more excerpts ...]\n\n", prompt[-300:])

2568 tokens

Answer the question using ONLY the numbered filing excerpts below. After your answer, on a new line, write 'CITED: ' followed by a comma-separated list of the excerpt numbers you used (e.g. 'CITED: 0,2'). If the excerpts don't contain enough information to answer, say so explicitly and write 'CITED: ' with nothing after it.

Excerpts:
[0] (Apple Inc., Item 7.    Management's Discussion and Analysis of Financial Condition and Results of Operat > Greater China, FY2024)
Greater China
Greater China net sales decreased during 2024 compared to 2023 due primarily to lower net sales of iPhone and iPad. The weakness in the renminbi relative to the U.S. dollar had an unfavorable year-over-year impact  

[... more excerpts ...]

 )
Japan
Japan net sales increased during 2024 compared to 2023 due primarily to higher net sales of iPhone. The weakness in the yen relative to the U.S. dollar had an unfavorable year-over-year impact on Japan net sales during 2024.

Question: Why did Greate

### Step 6 · Call the LLM

`complete_with_usage()` sends the prompt to the provider in `.env` (`LLM_PROVIDER`) and returns the reply text plus token counts and an estimated cost. Look at the last line of the reply: the `CITED:` line.

In [14]:
from core.llm import complete_with_usage

response = complete_with_usage(prompt)
print(response.text)
print(f"\n{llm.PROVIDER}/{llm.MODEL}: {response.input_tokens} tokens in, {response.output_tokens} out, ≈ ${response.cost_usd:.4f}")

Greater China net sales decreased during fiscal 2024 compared to 2023 primarily due to:
1. Lower net sales of iPhone and iPad.
2. The unfavorable year-over-year impact of the weakness in the renminbi relative to the U.S. dollar.

CITED: 0

gemini/gemini-3.5-flash: 3138 tokens in, 68 out, ≈ $0.0011


### Step 7 · Parse `CITED:` into citations

`_parse_response()` finds the final `CITED:` line with a regular expression, removes it from the answer, and maps each number back to its chunk. Numbers that don't match a chunk are ignored, so the model can't cite something it wasn't shown.

In [15]:
from core.rag import _parse_response

answer, citations = _parse_response(response.text, chunks)
print("ANSWER:", answer, "\n")
for c in citations:
    print(f"- {c.company_name} · {c.section[:60]} · FY{c.fiscal_year}")
    print(f"  “{c.text_snippet[:160].strip()}…”")

ANSWER: Greater China net sales decreased during fiscal 2024 compared to 2023 primarily due to:
1. Lower net sales of iPhone and iPad.
2. The unfavorable year-over-year impact of the weakness in the renminbi relative to the U.S. dollar. 

- Apple Inc. · Item 7.    Management's Discussion and Analysis of Financial · FY2024
  “Greater China
Greater China net sales decreased during 2024 compared to 2023 due primarily to lower net sales of iPhone and iPad. The weakness in the renminbi r…”


---
## Part 4 · The "not found" guardrail (step 8)

`answer_question()` runs steps 2–7 in one call. It refuses in two cases: the search returns nothing, or the LLM cites nothing. The second question isn't covered by this filing, so it should be refused instead of answered from the model's memory.

In [16]:
from core.rag import answer_question

for q in [QUESTION, "What was Apple's revenue in fiscal year 2019?"]:
    result = answer_question(q)
    print(f"Q: {q}\nA: {result.answer}")
    print(f"   {len(result.citations)} citation(s): {[c.section[:40] for c in result.citations]}\n")

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Q: Why did Greater China net sales decrease in fiscal 2024?
A: Greater China net sales decreased during 2024 compared to 2023 due primarily to lower net sales of iPhone and iPad. Additionally, the weakness in the renminbi relative to the U.S. dollar had an unfavorable year-over-year impact on Greater China net sales during 2024.
   1 citation(s): ["Item 7.    Management's Discussion and A"]



Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Q: What was Apple's revenue in fiscal year 2019?
A: I couldn't find information about this in the available filings.
   0 citation(s): []



---
## Part 5 · Recap and your turn

**What happened:** the question was searched by meaning and by keywords, the two lists were fused by rank, the best 8 chunks went into a strict prompt, and the model's `CITED:` line became the citations. No citation, no answer.

**Your turn:** change the question and run the cell. Ideas:
- *Which geographic segment had the largest revenue decline in 2024?*
- *What cybersecurity risks does Apple disclose in Item 1C?*
- Change `TOP_K` at the top of Part 2 to 3, re-run, and see whether the answer changes.

In [17]:
result = answer_question("What was Apple's gross margin percentage in 2024 vs. 2023?")
print(result.answer)
for c in result.citations:
    print("-", c.section[:70])

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

In 2024, Apple's total gross margin percentage was 46.2%, compared to 44.1% in 2023. 

Broken down by category:
* **Products:** 37.2% in 2024 vs. 36.5% in 2023
* **Services:** 73.9% in 2024 vs. 70.8% in 2023
- Item 7.    Management's Discussion and Analysis of Financial Condition


**Next:** `03_models_and_experiments.ipynb` compares models, chunkers and retrieval strategies; `04_agentic_rag.ipynb` turns this pipeline into an agent.